# Fusion des données pour la création de data set

In [ ]:
import pandas as pd

## Gestion des fichiers météos

In [ ]:
meteo = pd.read_csv("./climat_45/climat_mensuel_45_total.csv", sep=";")

meteo = meteo[["AAAAMM", "RR", "TMM", "ETP","LAT","LON"]]
meteo = meteo.rename(columns={
    "AAAAMM": "time",
    "RR": "PRELIQ_Q",
    "TMM": "T_Q",
    "LAT": "lat",
    "LON": "lon"
})

meteo["time"] = pd.to_datetime(meteo["time"].astype(str), format="%Y%m")

meteo_month = meteo.sort_values("time").reset_index(drop=True)
print(meteo_month.head())

## Gestion des fichiers ades

In [ ]:
nappe = pd.read_csv("./nappe_piezometres/nappe_02927X1013_P.csv", sep=";")

# Renommer la date
nappe = nappe.rename(columns={
    "date_mesure": "time"
})

# Conversion en datetime
nappe["time"] = pd.to_datetime(nappe["time"])

# Passage en mois
nappe["month"] = nappe["time"].dt.to_period("M")

# Agrégation mensuelle par piézomètre
nappe_month = nappe.groupby(["code_bss", "month"]).agg({
    "niveau_nappe_eau": "mean",  # niveau moyen mensuel
    "lon": "first",              # coordonnée fixe
    "lat": "first"
}).reset_index()

# Revenir à une date classique (premier jour du mois)
nappe_month["time"] = nappe_month["month"].dt.to_timestamp()

# Nettoyage
nappe_month = nappe_month.drop(columns="month")

print(nappe_month.head())

## Gestion des fichiers ETP

In [ ]:
import pandas as pd

etp = pd.read_csv("./etp_fao_hargreaves/etp_france_total.csv", sep=";")

# Conversion date (format 19700101)
etp["time"] = pd.to_datetime(etp["DATE"].astype(str), format="%Y%m%d")

etp = etp.rename(columns={
    "lat_dg": "lat",
    "lon_dg": "lon"
})

# Passage en mois
etp["month"] = etp["time"].dt.to_period("M")

# Agrégation mensuelle par maille (lat/lon)
etp_month = etp.groupby(["lat", "lon", "month"]).agg({
    "ETP_Q_H0175": "sum"   # ETP mensuelle = somme des jours
}).reset_index()

# Revenir en datetime
etp_month["time"] = etp_month["month"].dt.to_timestamp()

# Nettoyage
etp_month = etp_month.drop(columns=["month"])

print(etp_month.head())


## Fusion

In [ ]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np

def nearest_point(df_points, df_target):
    """
    Pour chaque ligne de df_target, trouver la ligne la plus proche dans df_points
    et retourner les indices correspondants.
    """
    tree = cKDTree(df_points[["lat", "lon"]].values)
    dist, idx = tree.query(df_target[["lat", "lon"]].values)
    return idx

# Copie de nappe
df = nappe_month.copy()

# Dates présentes dans nappe
dates = df["time"].unique()

# Initialiser les colonnes ETP et météo
df["ETP_Q_H0175"] = np.nan
df["PRELIQ_Q"] = np.nan
df["T_Q"] = np.nan

# Pour chaque date, chercher les points les plus proches
for d in dates:
    mask = df["time"] == d
    
    # Sous-ensembles pour cette date
    df_day = df[mask]
    etp_day = etp_month[etp_month["time"] == d]
    meteo_day = meteo_month[meteo_month["time"] == d]
    
    # Fusion ETP
    if len(etp_day) > 0:
        idx_etp = nearest_point(etp_day, df_day)
        df.loc[mask, "ETP_Q_H0175"] = etp_day.iloc[idx_etp]["ETP_Q_H0175"].values
    
    # Fusion Météo
    if len(meteo_day) > 0:
        idx_met = nearest_point(meteo_day, df_day)
        df.loc[mask, "PRELIQ_Q"] = meteo_day.iloc[idx_met]["PRELIQ_Q"].values
        df.loc[mask, "T_Q"] = meteo_day.iloc[idx_met]["T_Q"].values

# Export final en CSV
output_file = "./nappe_complete.csv"
df.to_csv(output_file, sep=";", index=False)

print(f"Fichier final créé : {output_file}")


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Liste des fichiers CSV dans le dossier
dossier = "../data"
fichiers = [f for f in os.listdir(dossier) if f.endswith(".csv")]

# Menu déroulant pour choisir un fichier
dropdown = widgets.Dropdown(
    options=fichiers,
    description='Fichier:',
)

# Bouton pour générer le graphique
button = widgets.Button(description="Afficher graphique")

# Widget pour afficher le print et le plot
out = widgets.Output()

# Fonction pour afficher le DataFrame et le graphique
def afficher_graphique(b):
    with out:
        clear_output()  # nettoie la sortie précédente
        fichier_choisi = dropdown.value
        df = pd.read_csv(os.path.join(dossier, fichier_choisi))
        print("Contenu du fichier :", fichier_choisi)
        print(df)  # ton print fonctionne maintenant

        # Tracer automatiquement les colonnes numériques si elles existent
        df_numeric = df.select_dtypes(include=['number'])
        if not df_numeric.empty:
            df_numeric.plot()
            plt.show()
        else:
            print("Aucune colonne numérique à tracer !")

# Lier la fonction au bouton
button.on_click(afficher_graphique)

# Afficher les widgets et la sortie
display(dropdown, button, out)


Dropdown(description='Fichier:', options=('data_02923X0007_F.csv', 'etp.csv', 'meteo.csv', 'nappe.csv'), value…

Button(description='Afficher graphique', style=ButtonStyle())

Output()